# 03 — Function Calling

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 02_Modelos_PreTreinados

---

## O que você vai aprender

- O que é **function calling** e por que é a base dos agentes
- Como **definir ferramentas** (tools) para o LLM usar
- Como o modelo **decide** quando e qual ferramenta chamar
- Como **executar a função** e devolver o resultado ao modelo
- O ciclo completo: **pergunta → tool call → execução → resposta final**
- Construindo um **mini-agente** funcional ao final

---

> 💡 Function calling é o que transforma um LLM de "gerador de texto" em um **agente que age no mundo**.  
> É a ponte entre linguagem natural e código Python.

---

### Nota sobre o DeepSeek

O DeepSeek às vezes retorna tool calls num formato proprietário chamado **DSML** em vez do padrão OpenAI.  
O `shared/tool_runner.py` detecta e trata isso automaticamente — você não precisa se preocupar.

## Conceito: como funciona?

```
┌─────────────────────────────────────────────────────────┐
│  1. Você define funções: nome + descrição + parâmetros  │
│  2. Envia pergunta + lista de funções disponíveis       │
│  3. O modelo decide: responde direto OU chama função    │
│  4. Se chamar função: você executa o código Python      │
│  5. Devolve o resultado ao modelo                       │
│  6. Modelo gera resposta final usando o resultado       │
└─────────────────────────────────────────────────────────┘

Usuário → "Qual arquivo tem mais linhas no projeto?"
    ↓
LLM → {tool_call: contar_linhas(caminho="...")}
    ↓
Python → executa a função → retorna 842
    ↓
LLM → "O arquivo models.py tem 842 linhas, o maior do projeto."
```

## Setup

In [1]:
import sys, os, json, datetime, pathlib
sys.path.append(os.path.abspath('..'))

from shared.llm_factory import get_provider_info
from shared.tool_runner import executar_com_tools, ToolRunner

info = get_provider_info()
print(f"Provider : {info['provider']} | Modelo: {info['model']}")
print("tool_runner carregado — fallback DSML ativo.")

Provider : deepseek | Modelo: deepseek-chat
tool_runner carregado — fallback DSML ativo.


---
## 1. Definindo a primeira ferramenta

Uma ferramenta tem 3 partes:
- **`name`** — identificador único
- **`description`** — o LLM usa isso para decidir quando chamar
- **`parameters`** — schema JSON dos argumentos

In [2]:
# Definição que vai para o LLM
tool_calculadora = {
    "type": "function",
    "function": {
        "name": "calcular",
        "description": "Executa operações matemáticas básicas: soma, subtração, multiplicação e divisão.",
        "parameters": {
            "type": "object",
            "properties": {
                "operacao": {
                    "type": "string",
                    "enum": ["soma", "subtracao", "multiplicacao", "divisao"],
                    "description": "Tipo de operação matemática"
                },
                "a": {"type": "number", "description": "Primeiro número"},
                "b": {"type": "number", "description": "Segundo número"}
            },
            "required": ["operacao", "a", "b"]
        }
    }
}

# Implementação Python real
def calcular(operacao: str, a: float, b: float) -> dict:
    operacoes = {
        "soma"         : lambda x, y: x + y,
        "subtracao"    : lambda x, y: x - y,
        "multiplicacao": lambda x, y: x * y,
        "divisao"      : lambda x, y: x / y if y != 0 else "Erro: divisão por zero"
    }
    resultado = operacoes[operacao](a, b)
    return {"resultado": resultado, "expressao": f"{a} {operacao} {b} = {resultado}"}

print("Ferramenta definida:", tool_calculadora["function"]["name"])
print("Teste direto:", calcular("multiplicacao", 7, 8))

Ferramenta definida: calcular
Teste direto: {'resultado': 56, 'expressao': '7 multiplicacao 8 = 56'}


---
## 2. O ciclo completo com tool_runner

In [3]:
# Teste básico — com verbose=True você vê cada passo do ciclo
print("=" * 50)
r = executar_com_tools(
    pergunta="Quanto é 1547 multiplicado por 38?",
    tools=[tool_calculadora],
    funcoes={"calcular": calcular},
    verbose=True
)
print(f"\nResposta: {r}")

[iter 1] 1 tool call(s) — OpenAI
  -> calcular({'operacao': 'multiplicacao', 'a': 1547, 'b': 38})
     = {'resultado': 58786, 'expressao': '1547 multiplicacao 38 = 58786'}
[iter 2] Resposta final

Resposta: 1547 multiplicado por 38 é igual a **58.786**.


In [4]:
# Quando o LLM NÃO usa ferramenta
print("=" * 50)
r = executar_com_tools(
    pergunta="O que é overfitting em machine learning?",
    tools=[tool_calculadora],
    funcoes={"calcular": calcular},
    verbose=True
)
print(f"\nResposta: {r}")

[iter 1] Resposta direta

Resposta: Overfitting (ou sobreajuste) é um problema comum em machine learning que ocorre quando um modelo se ajusta muito bem aos dados de treinamento, mas tem baixa capacidade de generalização para novos dados.

**Características do overfitting:**
- O modelo "decora" os dados de treinamento, incluindo ruídos e detalhes irrelevantes
- Desempenho excelente nos dados de treinamento, mas ruim em dados de teste/validação
- O modelo se torna muito complexo para o problema em questão

**Causas comuns:**
- Modelo muito complexo (muitos parâmetros)
- Poucos dados de treinamento
- Treinamento por muitas épocas (iterações)

**Soluções para evitar overfitting:**
1. **Coleta de mais dados** - aumenta a diversidade do conjunto de treinamento
2. **Regularização** - penaliza pesos muito grandes (L1, L2)
3. **Dropout** - desativa aleatoriamente neurônios durante o treinamento (em redes neurais)
4. **Early stopping** - para o treinamento quando o desempenho na validação começ

---
## 3. Múltiplas ferramentas — o LLM escolhe a certa

In [5]:
# Ferramenta 2: info do sistema
tool_sistema = {
    "type": "function",
    "function": {
        "name": "info_sistema",
        "description": "Retorna informações do sistema: data/hora atual, versão do Python e diretório de trabalho.",
        "parameters": {
            "type": "object",
            "properties": {
                "incluir": {
                    "type": "array",
                    "items": {"type": "string", "enum": ["data", "python", "diretorio"]},
                    "description": "Quais informações incluir"
                }
            },
            "required": ["incluir"]
        }
    }
}

def info_sistema(incluir: list) -> dict:
    resultado = {}
    if "data"      in incluir: resultado["data_hora"]      = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    if "python"    in incluir: resultado["python_versao"]  = sys.version.split()[0]
    if "diretorio" in incluir: resultado["diretorio_atual"] = str(pathlib.Path.cwd())
    return resultado


# Ferramenta 3: listar arquivos
tool_listar_arquivos = {
    "type": "function",
    "function": {
        "name": "listar_arquivos",
        "description": "Lista arquivos de um diretório filtrando por extensão.",
        "parameters": {
            "type": "object",
            "properties": {
                "caminho" : {"type": "string", "description": "Caminho do diretório"},
                "extensao": {"type": "string", "description": "Extensão ex: .py .ipynb .md — use * para todos"}
            },
            "required": ["caminho", "extensao"]
        }
    }
}

def listar_arquivos(caminho: str, extensao: str) -> dict:
    p = pathlib.Path(caminho)
    if not p.exists():
        return {"erro": f"Diretório não encontrado: {caminho}"}
    padrao = f"*{extensao}" if extensao != "*" else "*"
    arquivos = [str(f.relative_to(p)) for f in p.rglob(padrao) if f.is_file()]
    return {"diretorio": caminho, "extensao": extensao, "total": len(arquivos), "arquivos": arquivos[:20]}


TODAS_AS_TOOLS   = [tool_calculadora, tool_sistema, tool_listar_arquivos]
TODAS_AS_FUNCOES = {"calcular": calcular, "info_sistema": info_sistema, "listar_arquivos": listar_arquivos}

print(f"{len(TODAS_AS_TOOLS)} ferramentas prontas.")

3 ferramentas prontas.


In [6]:
perguntas = [
    "Que horas são agora e qual versão do Python estou usando?",
    "Quantos notebooks .ipynb existem na pasta atual?",
    "Quanto é 2 elevado a 10? (use multiplicações sucessivas)",
]

for pergunta in perguntas:
    print(f"\n{'='*55}")
    print(f"Pergunta: {pergunta}")
    print('-'*55)
    r = executar_com_tools(pergunta, TODAS_AS_TOOLS, TODAS_AS_FUNCOES, verbose=True)
    print(f"\nResposta: {r}")


Pergunta: Que horas são agora e qual versão do Python estou usando?
-------------------------------------------------------
[iter 1] 1 tool call(s) — OpenAI
  -> info_sistema({'incluir': ['data', 'python']})
     = {'data_hora': '27/03/2026 09:12:39', 'python_versao': '3.10.19'}
[iter 2] Resposta final

Resposta: Agora são **09:12:39** do dia **27/03/2026** e você está usando a versão **Python 3.10.19**.

Pergunta: Quantos notebooks .ipynb existem na pasta atual?
-------------------------------------------------------
[iter 1] 1 tool call(s) — OpenAI
  -> info_sistema({'incluir': ['diretorio']})
     = {'diretorio_atual': 'C:\\Users\\Jorge Maques\\Documents\\Especialista_em_AI\\EAI_07_AI_Generative\\02_Modelos_PreTreinados'}
[iter 2] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'C:\\Users\\Jorge Maques\\Documents\\Especialista_em_AI\\EAI_07_AI_Generative\\02_Modelos_PreTreinados', 'extensao': '.ipynb'})
     = {'diretorio': 'C:\\Users\\Jorge Maques\\Documents\\Especialista

---
## 4. ToolRunner — interface orientada a objetos

Para projetos maiores, o `ToolRunner` permite registrar ferramentas progressivamente e reutilizar o agente.

In [7]:
# ToolRunner com encadeamento de registro
runner = (
    ToolRunner(system="Você é um assistente técnico. Responda em português.", verbose=True)
    .registrar(tool_calculadora,    calcular)
    .registrar(tool_sistema,        info_sistema)
    .registrar(tool_listar_arquivos, listar_arquivos)
)

print(runner)

r = runner.perguntar("Me dê um resumo do ambiente: data/hora, Python e quantos arquivos .py existem aqui.")
print(f"\nResposta:\n{r}")

ToolRunner(tools=['calcular', 'info_sistema', 'listar_arquivos'])
[iter 1] 1 tool call(s) — OpenAI
  -> info_sistema({'incluir': ['data', 'python', 'diretorio']})
     = {'data_hora': '27/03/2026 09:13:38', 'python_versao': '3.10.19', 'diretorio_atual': 'C:\\Users\\Jorge Maques\\Documents\\Especialista_em_AI\\EAI_07_AI_Generative\\02_Modelos_PreTreinados'}
[iter 2] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'C:\\Users\\Jorge Maques\\Documents\\Especialista_em_AI\\EAI_07_AI_Generative\\02_Modelos_PreTreinados', 'extensao': '.py'})
     = {'diretorio': 'C:\\Users\\Jorge Maques\\Documents\\Especialista_em_AI\\EAI_07_AI_Generative\\02_Modelos_PreTreinados', 'extensao': '.py', 'total': 0, 'arquivos': []}
[iter 3] Resposta final

Resposta:
## Resumo do Ambiente:

**Data/Hora:** 27 de março de 2026, 09:13:38

**Versão do Python:** 3.10.19

**Diretório atual:** `C:\Users\Jorge Maques\Documents\Especialista_em_AI\EAI_07_AI_Generative\02_Modelos_PreTreinados`

**Arquivos .py:** 0 a

---
## 5. Mini-agente do projeto ESPECIALISTA_EM_IA

In [8]:
tool_resumo_modulo = {
    "type": "function",
    "function": {
        "name": "resumo_modulo",
        "description": "Retorna informações sobre um módulo do projeto ESPECIALISTA_EM_IA.",
        "parameters": {
            "type": "object",
            "properties": {
                "modulo": {
                    "type": "string",
                    "enum": ["EAI_01","EAI_02","EAI_03","EAI_04","EAI_05","EAI_06","EAI_07","EAI_08"],
                    "description": "Código do módulo"
                }
            },
            "required": ["modulo"]
        }
    }
}

MODULOS_INFO = {
    "EAI_01": {"nome": "Fundamentos Matemáticos",  "topicos": ["Vetores", "Regressão Linear", "Álgebra Linear"],                              "status": "completo"},
    "EAI_02": {"nome": "Machine Learning",          "topicos": ["KNN", "SVM", "Random Forest", "Projetos: Diabetes, Alunos"],                "status": "completo"},
    "EAI_03": {"nome": "Deep Learning",              "topicos": ["ANN", "CNN", "LSTM", "GRU", "Projetos: ArtClassifier, Ações"],             "status": "completo"},
    "EAI_04": {"nome": "NLP Clássico",               "topicos": ["BoW", "TF-IDF", "Word Embeddings", "Projetos: FAQ, Feedback"],             "status": "completo"},
    "EAI_05": {"nome": "NLP com Transformers",       "topicos": ["Tokenização", "BERT", "Chatbot", "Classificação"],                        "status": "completo"},
    "EAI_06": {"nome": "Visão Computacional",        "topicos": ["OpenCV", "YOLOv5", "OCR", "Projetos: Reconhecimento Facial"],              "status": "completo"},
    "EAI_07": {"nome": "IA Generativa",              "topicos": ["LLMs", "Prompt Engineering", "Function Calling", "RAG", "Agentes"],        "status": "em andamento"},
    "EAI_08": {"nome": "MLOps e Implantação",        "topicos": ["Docker", "CI/CD", "Monitoramento", "Projetos: Fraudes, Diabetes"],         "status": "pendente"},
}

def resumo_modulo(modulo: str) -> dict:
    info = MODULOS_INFO.get(modulo)
    if not info:
        return {"erro": f"Módulo {modulo} não encontrado"}
    return {"modulo": modulo, **info, "total_topicos": len(info["topicos"])}


SYSTEM_AGENTE = """Você é um assistente técnico do projeto ESPECIALISTA_EM_IA de Carlos Henrique.
Use as ferramentas para consultar informações concretas do projeto.
Responda sempre em português."""

agente = (
    ToolRunner(system=SYSTEM_AGENTE, verbose=False)
    .registrar(tool_resumo_modulo,   resumo_modulo)
    .registrar(tool_sistema,         info_sistema)
)

print(agente)

ToolRunner(tools=['resumo_modulo', 'info_sistema'])


In [9]:
perguntas = [
    "O que foi estudado no módulo EAI_03?",
    "Quais módulos já estão completos?",
    "Qual é a data de hoje?",
]

for pergunta in perguntas:
    print(f"\n👤 {pergunta}")
    print(f"🤖 {agente.perguntar(pergunta)}")


👤 O que foi estudado no módulo EAI_03?
🤖 No módulo **EAI_03 - Deep Learning**, foram estudados os seguintes tópicos:

## **Tópicos abordados:**
1. **ANN** - Redes Neurais Artificiais
2. **CNN** - Redes Neurais Convolucionais
3. **LSTM** - Long Short-Term Memory
4. **GRU** - Gated Recurrent Units

## **Projetos práticos desenvolvidos:**
- **ArtClassifier** - Classificador de arte usando técnicas de Deep Learning
- **Ações** - Projeto relacionado a análise e previsão de ações do mercado financeiro

## **Status:** Completo ✅
**Total de tópicos:** 5

Este módulo focou nas principais arquiteturas de Deep Learning, incluindo redes neurais tradicionais (ANN), redes convolucionais para processamento de imagens (CNN), e redes recorrentes para processamento de sequências temporais (LSTM e GRU). Os projetos práticos permitiram aplicar esses conceitos em problemas reais como classificação de arte e análise de dados financeiros.

👤 Quais módulos já estão completos?
🤖 

👤 Qual é a data de hoje?
🤖 H

---
## Resumo

| Conceito | O que aprendemos |
|---|---|
| **Definição de tool** | `name` + `description` + `parameters` (JSON schema) |
| **Ciclo completo** | pergunta → tool_call → execução Python → resposta final |
| **Fallback DSML** | `tool_runner.py` trata automaticamente o formato proprietário do DeepSeek |
| **ToolRunner** | Interface OO para registrar tools e reusar o agente |
| **Mini-agente** | System prompt + múltiplas tools + perguntas em linguagem natural |

---